# ML Random Forest - Classification pour Trading

**Objectif**: Utiliser les forets aleatoires (Random Forest) pour predire la direction des prix.

## Strategie

1. **Ensemble de decision trees**: Agreger les predictions de multiples arbres
2. **Classification binaire**: Predire si le prix va monter ou descendre
3. **Features techniques**: RSI, MACD, Bollinger, momentum, volume
4. **Feature importance**: Identifier les indicateurs les plus predictifs
5. **Walk-forward validation**: Re-entrainement periodique

## Prerequis

- scikit-learn: `pip install scikit-learn`
- Comprehension des arbres de decision et bagging

## Duree estimee: 45 minutes

In [ ]:
# Initialisation QuantBook
from AlgorithmImports import *

qb = QuantBook()

# Periode d'analyse
qb.SetStartDate(2020, 1, 1)
qb.SetEndDate(2024, 12, 31)

print(f"Periode: {qb.StartDate} a {qb.EndDate}")

## 1. Chargement des Donnees

In [ ]:
# Ajouter les actifs
tickers = ['SPY', 'QQQ', 'IWM', 'TLT', 'GLD']
symbols = {}

for ticker in tickers:
    symbols[ticker] = qb.AddEquity(ticker, Resolution.DAILY).Symbol

# Recuperer les donnees
history = qb.History(list(symbols.values()), 365*5, Resolution.Daily)

closes = history['close'].unstack(level=0)
volumes = history['volume'].unstack(level=0)
highs = history['high'].unstack(level=0)
lows = history['low'].unstack(level=0)

print(f"Donnees: {closes.shape[0]} jours, {closes.shape[1]} actifs")
closes.head()

## 2. Feature Engineering

In [ ]:
def calculate_features(closes, volumes, highs, lows):
    """Calcule les features techniques pour Random Forest."""
    features = pd.DataFrame()
    
    for ticker in closes.columns:
        close = closes[ticker]
        volume = volumes[ticker]
        high = highs[ticker]
        low = lows[ticker]
        
        returns = close.pct_change()
        
        # Moving averages
        sma_5 = close.rolling(5).mean()
        sma_10 = close.rolling(10).mean()
        sma_20 = close.rolling(20).mean()
        sma_50 = close.rolling(50).mean()
        
        # RSI
        delta = close.diff()
        gain = (delta.where(delta > 0, 0)).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        rs = gain / loss
        rsi = 100 - (100 / (1 + rs))
        
        # MACD
        ema_12 = close.ewm(span=12).mean()
        ema_26 = close.ewm(span=26).mean()
        macd = ema_12 - ema_26
        macd_signal = macd.ewm(span=9).mean()
        
        # Bollinger Bands
        bb_middle = close.rolling(20).mean()
        bb_std = close.rolling(20).std()
        bb_position = (close - bb_middle) / (2 * bb_std)
        
        # Stochastic
        stoch_k = 100 * (close - low.rolling(14).min()) / (high.rolling(14).max() - low.rolling(14).min() + 0.0001)
        
        # Momentum
        mom_5 = close / close.shift(5) - 1
        mom_10 = close / close.shift(10) - 1
        mom_20 = close / close.shift(20) - 1
        
        # Volatility
        vol_10 = returns.rolling(10).std()
        vol_20 = returns.rolling(20).std()
        
        # Volume
        volume_sma = volume.rolling(20).mean()
        volume_ratio = volume / (volume_sma + 0.0001)
        
        # Price ratios
        price_sma5 = close / sma_5
        price_sma20 = close / sma_20
        price_sma50 = close / sma_50
        
        # Combiner
        ticker_features = pd.DataFrame({
            f'{ticker}_rsi': rsi,
            f'{ticker}_bb_position': bb_position,
            f'{ticker}_macd': macd,
            f'{ticker}_macd_hist': macd - macd_signal,
            f'{ticker}_stoch_k': stoch_k,
            f'{ticker}_mom_5': mom_5,
            f'{ticker}_mom_10': mom_10,
            f'{ticker}_mom_20': mom_20,
            f'{ticker}_vol_10': vol_10,
            f'{ticker}_vol_20': vol_20,
            f'{ticker}_volume_ratio': volume_ratio,
            f'{ticker}_price_sma5': price_sma5,
            f'{ticker}_price_sma20': price_sma20,
            f'{ticker}_price_sma50': price_sma50,
            f'{ticker}_sma_ratio_5_20': sma_5 / sma_20,
            f'{ticker}_sma_ratio_20_50': sma_20 / sma_50,
        })
        
        features = pd.concat([features, ticker_features], axis=1)
    
    return features.fillna(0)

# Calculer les features
features = calculate_features(closes, volumes, highs, lows)

print(f"Features shape: {features.shape}")
print(f"\nNombre de features par actif: {features.shape[1] // len(tickers)}")

## 3. Preparation des Donnees d'Entrainement

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Preparer les donnees pour SPY
spy_features = features[[c for c in features.columns if 'SPY' in c]]
spy_returns = closes['SPY'].pct_change()

# Target: direction (1 si hausse, 0 si baisse)
target = (spy_returns.shift(-1) > 0).astype(int)

# Creer le dataset
data = spy_features.copy()
data['target'] = target
data = data.dropna()

X = data.drop('target', axis=1)
y = data['target']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=False  # Pas de shuffle pour series temporelles
)

# Normaliser (MinMaxScaler pour Random Forest)
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"\nDistribution des classes (train):")
print(y_train.value_counts(normalize=True))

## 4. Entrainement Random Forest

In [ ]:
# Entrainer le modele
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)

# Previsions
y_pred = rf_model.predict(X_test_scaled)
y_proba = rf_model.predict_proba(X_test_scaled)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")
print(f"\nRapport de classification:")
print(classification_report(y_test, y_pred, target_names=['Baisse', 'Hausse']))

## 5. Feature Importance

In [ ]:
import matplotlib.pyplot as plt

# Importance des features
importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

# Afficher top 10
plt.figure(figsize=(12, 6))
plt.barh(importance['feature'].head(10), importance['importance'].head(10))
plt.xlabel('Importance')
plt.title('Top 10 Features - Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 10 features les plus importantes:")
print(importance.head(10))

## 6. Matrice de Confusion

In [ ]:
import seaborn as sns

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Baisse', 'Hausse'],
            yticklabels=['Baisse', 'Hausse'])
plt.xlabel('Prediction')
plt.ylabel('Reel')
plt.title('Matrice de Confusion - Random Forest')
plt.tight_layout()
plt.show()

## 7. Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

# Grille de parametres
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Recherche (sur un sous-ensemble pour la vitesse)
rf_grid = RandomForestClassifier(random_state=42, n_jobs=-1)

grid_search = GridSearchCV(
    rf_grid, param_grid, cv=3, scoring='accuracy', n_jobs=-1, verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print(f"\nMeilleurs parametres: {grid_search.best_params_}")
print(f"Meilleur score: {grid_search.best_score_:.2%}")

## 8. Walk-Forward Validation

In [ ]:
def walk_forward_validation(X, y, train_size=252, test_size=63):
    """Validation walk-forward pour series temporelles."""
    
    predictions = []
    actuals = []
    dates = []
    
    for start in range(0, len(X) - train_size - test_size, test_size):
        train_end = start + train_size
        test_end = train_end + test_size
        
        # Split
        X_train_wf = X.iloc[start:train_end]
        y_train_wf = y.iloc[start:train_end]
        X_test_wf = X.iloc[train_end:test_end]
        y_test_wf = y.iloc[train_end:test_end]
        
        # Normaliser
        scaler_wf = MinMaxScaler()
        X_train_scaled = scaler_wf.fit_transform(X_train_wf)
        X_test_scaled = scaler_wf.transform(X_test_wf)
        
        # Entrainer
        model = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        model.fit(X_train_scaled, y_train_wf)
        
        # Prevoir
        pred = model.predict(X_test_scaled)
        predictions.extend(pred)
        actuals.extend(y_test_wf.values)
        dates.extend(X_test_wf.index)
    
    return np.array(predictions), np.array(actuals), dates

# Executer walk-forward
wf_pred, wf_actual, wf_dates = walk_forward_validation(X, y)

# Resultats
wf_accuracy = (wf_pred == wf_actual).mean()
direction_accuracy = ((wf_pred == 1) == (wf_actual == 1)).mean()

print(f"Walk-Forward Accuracy: {wf_accuracy:.2%}")
print(f"Direction Accuracy: {direction_accuracy:.2%}")

## 9. Backtest de la Strategie

In [ ]:
def backtest_rf_strategy(closes, features, train_period=252, rebalance_freq=5):
    """Backtest strategie Random Forest."""
    
    portfolio_value = 100000
    positions = {}
    trade_log = []
    
    for i in range(train_period, len(closes) - 1, rebalance_freq):
        current_date = closes.index[i]
        
        # Entrainer le modele
        train_features = features.iloc[i-train_period:i]
        spy_returns = closes['SPY'].pct_change().iloc[i-train_period:i]
        target = (spy_returns.shift(-1) > 0).astype(int)
        
        train_data = train_features.copy()
        train_data['target'] = target
        train_data = train_data.dropna()
        
        if len(train_data) < 50:
            continue
        
        X_train = train_data.drop('target', axis=1)
        y_train = train_data['target']
        
        scaler = MinMaxScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        model = RandomForestClassifier(
            n_estimators=100, max_depth=10, random_state=42, n_jobs=-1
        )
        model.fit(X_train_scaled, y_train)
        
        # Prevoir pour chaque ETF
        predictions = {}
        for ticker in tickers:
            ticker_features = features[[c for c in features.columns if ticker in c]]
            
            if len(ticker_features) < i + 1:
                continue
            
            latest = ticker_features.iloc[i:i+1].values
            latest_scaled = scaler.transform(latest)
            
            proba = model.predict_proba(latest_scaled)[0]
            predictions[ticker] = proba[1]  # Probabilite de hausse
        
        # Liquider positions existantes
        for t, qty in positions.items():
            portfolio_value += qty * closes[t].iloc[i]
        positions = {}
        
        # Nouvelles positions (top 2 probabilites > 0.55)
        sorted_preds = sorted(predictions.items(), key=lambda x: x[1], reverse=True)
        
        count = 0
        for ticker, prob in sorted_preds:
            if prob > 0.55 and count < 2:
                position_size = portfolio_value * 0.45
                price = closes[ticker].iloc[i]
                positions[ticker] = position_size / price
                portfolio_value -= position_size
                count += 1
    
    # Valeur finale
    final_value = portfolio_value
    for t, qty in positions.items():
        final_value += qty * closes[t].iloc[-1]
    
    return {
        'initial': 100000,
        'final': final_value,
        'return': (final_value - 100000) / 100000
    }

# Executer le backtest
results = backtest_rf_strategy(closes, features)

print(f"\nBacktest Results:")
print(f"Valeur initiale: ${results['initial']:,.2f}")
print(f"Valeur finale: ${results['final']:,.2f}")
print(f"Rendement total: {results['return']:.2%}")

## 10. Avantages et Limites de Random Forest

### Avantages
- **Robustesse**: Agrege plusieurs arbres, reduit le surapprentissage
- **Feature importance**: Interpretable, identifie les indicateurs cles
- **Non-lineaire**: Capture les relations complexes sans hypothese lineaire
- **Peu de preprocessing**: Pas besoin de standardisation stricte

### Limites
- **Lenteur**: Peut etre lent avec beaucoup d'arbres
- **Memoire**: Stocke tous les arbres
- **Extrapolation**: Mauvais sur donnees hors distribution d'entrainement
- **Lag**: Les decisions basees sur les features passees peuvent etre en retard